In [27]:
# buckets for activation values

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from Data.MNIST_Loader import load_and_preprocess_data

# Load model
model = tf.keras.models.load_model("Models/best_model.h5")

# Load a batch of inputs
ds_train, ds_val, ds_test = load_and_preprocess_data()
for batch in ds_val.take(1):
    inputs, _ = batch  # shape: (batch_size, 28, 28, 1)

# Identify the flatten layer and create an inference function for it
flatten_layer = None
for layer in model.layers:
    if 'flatten' in layer.name.lower():
        flatten_layer = layer
        break

if flatten_layer is None:
    raise ValueError("No Flatten layer found in the model.")

# Use the functional API to create a new model from the original model's input tensors to the flatten layer output
flatten_model = Model(inputs=model.inputs, outputs=flatten_layer.output)

# Run inference to get flattened activations
a = flatten_model.predict(inputs)

# Identify the first Dense layer to get weights and biases
first_dense_layer = None
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        first_dense_layer = layer
        break

if first_dense_layer is None:
    raise ValueError("No Dense layer found in the model.")

kernel, bias = first_dense_layer.get_weights()

# Compute pre-activation z = wa + b
z = np.dot(a, kernel) + bias

# Flatten z and bucket as desired
z_flat = z.flatten()

bucket_edges = np.concatenate(([-np.inf, -1], np.arange(-1, 1, 0.1), [1, np.inf]))
bucket_labels = ['Less than -1'] + \
                [f"{round(left,1)} to {round(left+0.1,1)}" for left in np.arange(-1,1,0.1)] + \
                ['Greater than 1']

bucket_indices = np.digitize(z_flat, bucket_edges) - 1
counts = {label: np.sum(bucket_indices == i) for i, label in enumerate(bucket_labels)}

for label in bucket_labels:
    print(f"{label}: {counts[label]}")


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Less than -1: 0
-1.0 to -0.9: 0
-0.9 to -0.8: 0
-0.8 to -0.7: 0
-0.7 to -0.6: 0
-0.6 to -0.5: 0
-0.5 to -0.4: 0
-0.4 to -0.3: 0
-0.3 to -0.2: 13
-0.2 to -0.1: 328
-0.1 to -0.0: 5184
-0.0 to 0.1: 29857
0.1 to 0.2: 25056
0.2 to 0.3: 3468
0.3 to 0.4: 92
0.4 to 0.5: 2
0.5 to 0.6: 0
0.6 to 0.7: 0
0.7 to 0.8: 0
0.8 to 0.9: 0
0.9 to 1.0: 0
Greater than 1: 0


2025-10-13 09:59:40.943608: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-10-13 09:59:40.943753: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [28]:
# Check weight range

import numpy as np
import tensorflow as tf
from Data.MNIST_Loader import load_and_preprocess_data

# Load model
model = tf.keras.models.load_model("Models/best_model.h5")

# Identify the first Dense layer to get weights and biases
first_dense_layer = None
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        first_dense_layer = layer
        break

if first_dense_layer is None:
    raise ValueError("No Dense layer found in the model.")

kernel, bias = first_dense_layer.get_weights()

# Flatten all weights and biases together for analysis
all_weights = np.concatenate([kernel.flatten(), bias.flatten()])

# Define bucket edges
inner_edges = np.linspace(-0.01, 0.01, 21)  # 20 buckets in [-0.01, 0.01]
bucket_edges = np.concatenate(([-np.inf], inner_edges, [np.inf]))

# Create bucket labels:
bucket_labels = ['<-0.01']
bucket_labels += [
    f"{round(inner_edges[i],5)} to {round(inner_edges[i+1],5)}"
    for i in range(len(inner_edges)-1)
]
bucket_labels += ['>0.01']

bucket_indices = np.digitize(all_weights, bucket_edges) - 1  # digitize is 1-based, so subtract 1
counts = {label: np.sum(bucket_indices == i) for i, label in enumerate(bucket_labels)}

for label in bucket_labels:
    print(f"{label}: {counts[label]}")


<-0.01: 4243
-0.01 to -0.009: 1640
-0.009 to -0.008: 2341
-0.008 to -0.007: 3737
-0.007 to -0.006: 5913
-0.006 to -0.005: 10145
-0.005 to -0.004: 17314
-0.004 to -0.003: 29159
-0.003 to -0.002: 46303
-0.002 to -0.001: 72821
-0.001 to 0.0: 203984
0.0 to 0.001: 203970
0.001 to 0.002: 69280
0.002 to 0.003: 43793
0.003 to 0.004: 27767
0.004 to 0.005: 16757
0.005 to 0.006: 9945
0.006 to 0.007: 6126
0.007 to 0.008: 3609
0.008 to 0.009: 2219
0.009 to 0.01: 1404
>0.01: 2530


In [32]:
import numpy as np
import tensorflow as tf
from Data.MNIST_Loader import load_and_preprocess_data

# Load model
model = tf.keras.models.load_model("Models/best_model.h5")

# Identify Dense layers
dense_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Dense)]

if len(dense_layers) < 2:
    raise ValueError("Model has fewer than 2 Dense layers.")

# Get weights and biases for first and second Dense layers
kernel_1, bias_1 = dense_layers[0].get_weights()
kernel_2, bias_2 = dense_layers[1].get_weights()

# Compute average of weights and biases for each layer
avg_weight_1 = np.mean(np.concatenate([kernel_1.flatten(), bias_1.flatten()]))
avg_weight_2 = np.mean(np.concatenate([kernel_2.flatten(), bias_2.flatten()]))

print(f"Average weight value in first Dense layer: {avg_weight_1}")
print(f"Average weight value in second Dense layer: {avg_weight_2}")

# Print weights of first 10 neurons in first Dense layer (kernel columns)
# print("\nWeights of first 10 neurons in first Dense layer:")
# for i in range(10):
#     neuron_weights = kernel_1[:, i]
#     print(f"Neuron {i+1}: {neuron_weights}")

# # Print weights of first 10 neurons in second Dense layer (kernel columns)
# print("\nWeights of first 10 neurons in second Dense layer:")
# for i in range(10):
#     neuron_weights = kernel_2[:, i]
#     print(f"Neuron {i+1}: {neuron_weights}")


Average weight value in first Dense layer: -5.9902871726080775e-05
Average weight value in second Dense layer: -0.0003211995935998857
